
# SonarVision - Model Training (Google Colab / YOLOv8s)

This notebook **downloads both datasets from Hugging Face inside Colab**, extracts them,
trains a YOLOv8s detector, and saves everything to your **Google Drive** so it survives
Colab session resets.

- **Filtered images** (1.5 GB):  `lalitchandra00/sonar_filtered_dataset` -> `noise_filtered_training.zip`
- **Labels** (extracted from the 6 GB original):  `lalitchandra00/sonar_dataset` -> `dataset_final.zip`
- Images are **only** taken from the filtered zip; labels are matched to them by filename (verified 1-to-1).
- The noise filter only upscales square 640px images to square 1024px, so the bounding boxes stay valid.

## Where things are stored on Drive  (`MyDrive/SIH/SonarVision/backend/`)

| Path                                   | Contents                                                                |
|----------------------------------------|-------------------------------------------------------------------------|
| `backend/runs/dataset_yolov8/`         | full run: `results.csv` (epoch metrics), accuracy graphs, confusion matrix, weights |
| `backend/training_checkpoints/`        | `last.pt` + `state.json` -> auto-resume when a Colab session dies        |
| `backend/best/`                        | final `best.pt`, exported `best.onnx`, `training_summary.txt`            |

> Run all cells top-to-bottom. Training takes several hours, but if Colab times out you
> simply **re-run the notebook** - it auto-resumes from the last saved epoch.



## 1. Configuration
Edit only this cell if you need to.


In [ ]:

# =============================== CONFIG ===============================
MODEL_SIZE       = 's'        # 'n' nano | 's' small (default, T4) | 'm' medium
EPOCHS           = 120        # training budget - early stopping stops sooner if converged
PATIENCE         = 30         # stop if val mAP50-95 does not improve for this many epochs
IMGSZ            = 640        # input size (640 = fast + safe on T4)
BATCH            = 16         # reduce to 8 if you hit CUDA out-of-memory
WORKERS          = 2
SEED             = 42

RESUME           = True       # auto-resume from backend/training_checkpoints/last.pt
FORCE_CONTINUE   = False      # True = resume even if a previous run already finished (extends it)

CACHE_ZIPS_ON_DRIVE = False   # True keeps the ~7.5 GB zips on Drive (else re-download each session)

# Expected dataset sizes (sanity check after extraction)
TRAIN_EXPECTED   = 8004       # 1334 images x 6 classes
VAL_EXPECTED     = 198        # 33 images x 6 classes

# --- Google Drive target -----------------------------------------------------------------
DRIVE_MOUNT   = '/content/drive'
DRIVE_PROJECT = 'SIH/SonarVision'   # -> MyDrive/SIH/SonarVision/backend

# --- Hugging Face datasets ---------------------------------------------------------------
HF_FILTERED = 'https://huggingface.co/datasets/lalitchandra00/sonar_filtered_dataset/resolve/main/noise_filtered_training.zip'
HF_ORIGINAL = 'https://huggingface.co/datasets/lalitchandra00/sonar_dataset/resolve/main/dataset_final.zip'



## 2. Mount Google Drive (checkpoints/runs must survive Colab resets)


In [ ]:

from pathlib import Path
import os, shutil, json

try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
except Exception as e:
    print('Not in Colab (or Drive mount failed):', e)

MYDRIVE = Path(DRIVE_MOUNT) / 'MyDrive'
if not MYDRIVE.exists():
    MYDRIVE = Path.home() / 'SonarVision_drive'     # small local fallback
    MYDRIVE.mkdir(parents=True, exist_ok=True)

BACKEND        = MYDRIVE / DRIVE_PROJECT / 'backend'
RUN_DRIVE      = BACKEND / 'runs'
RUN_DIR_DRIVE  = RUN_DRIVE / 'dataset_yolov8'
CKPT_DIR       = BACKEND / 'training_checkpoints'
BEST_DIR       = BACKEND / 'best'
for d in (RUN_DRIVE, RUN_DIR_DRIVE, CKPT_DIR, BEST_DIR):
    d.mkdir(parents=True, exist_ok=True)

CKPT = CKPT_DIR / 'last.pt'
DONE = CKPT_DIR / 'DONE'

# Local session paths - fast Colab disk, wiped when the runtime resets
WORK          = Path('/content/sonar_work')
DATASET       = WORK / 'dataset'
ZIPS          = WORK / 'zips'
RUNS_LOCAL    = WORK / 'runs'
RUN_DIR_LOCAL = RUNS_LOCAL / 'dataset_yolov8'
for d in (WORK, DATASET, ZIPS, RUNS_LOCAL):
    d.mkdir(parents=True, exist_ok=True)

print('BACKEND (Drive):', BACKEND)
print('Checkpoint     :', CKPT)
print('Dataset (local):', DATASET)



## 3. Install dependencies & check GPU


In [ ]:

%pip install -q --no-cache-dir "ultralytics>=8.2.40"

import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import requests
import zipfile
from collections import Counter, defaultdict
from ultralytics import YOLO
from tqdm.auto import tqdm

print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    DEVICE = 0
    print('GPU:', torch.cuda.get_device_name(0))
else:
    DEVICE = 'cpu'
    print('WARNING: no GPU found - CPU training would be extremely slow.')



## 4. Download & extract the datasets (from Hugging Face, inside Colab)

- `noise_filtered_training.zip` -> filtered **images** (train/val)
- `dataset_final.zip` -> only the **label .txt files** are extracted (tiny)


In [ ]:

def download_to(url, dst, desc='download'):
    '''Stream a file to disk, skipping it if it already exists.'''
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size > 1_000_000:
        print('Already downloaded:', dst)
        return dst
    r = requests.get(url, stream=True, timeout=(10, 600))
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    tmp = dst.with_suffix('.part')
    with open(tmp, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True, desc=desc) as pbar:
        for chunk in r.iter_content(chunk_size=1 << 20):
            f.write(chunk)
            pbar.update(len(chunk))
    tmp.replace(dst)
    print('Downloaded:', dst)
    return dst

def extract_images(zip_path, subdir_map):
    '''subdir_map = {zip_subdir_name: destination_dir} (image files only).'''
    with zipfile.ZipFile(zip_path) as z:
        for zsub, dest in subdir_map.items():
            dest.mkdir(parents=True, exist_ok=True)
            got = 0
            for m in tqdm(z.namelist(), desc='extract ' + zsub):
                if zsub in m.split('/') and m.lower().endswith(('.jpg', '.jpeg', '.png')):
                    tgt = dest / m.rsplit('/', 1)[-1]
                    if not tgt.exists():
                        with z.open(m) as src, tgt.open('wb') as dst:
                            shutil.copyfileobj(src, dst)
                    got += 1
            print(zsub, '->', got, 'files ->', dest)

print('Step A: filtered images ...')
zf = download_to(HF_FILTERED, ZIPS / 'noise_filtered_training.zip', 'filtered images (1.5 GB)')
extract_images(zf, {
    'train_filtered': DATASET / 'train' / 'images',
    'val_filtered':   DATASET / 'val' / 'images',
})
ntr = len(list((DATASET / 'train' / 'images').glob('*.jpg')))
nva = len(list((DATASET / 'val' / 'images').glob('*.jpg')))
print(f'train images: {ntr}  (expected {TRAIN_EXPECTED})')
print(f'val images  : {nva}  (expected {VAL_EXPECTED})')
assert ntr == TRAIN_EXPECTED, f'train image count mismatch: {ntr} != {TRAIN_EXPECTED}'
assert nva == VAL_EXPECTED, f'val image count mismatch: {nva} != {VAL_EXPECTED}'
print('Images OK.')


In [ ]:

print('Step B: labels (from the original 6 GB dataset zip) ...')
zo = download_to(HF_ORIGINAL, ZIPS / 'dataset_final.zip', 'original dataset (6 GB, labels only needed)')
with zipfile.ZipFile(zo) as z:
    for sub in ('train', 'val'):
        dest = DATASET / sub / 'labels'
        dest.mkdir(parents=True, exist_ok=True)
        got = 0
        for m in tqdm(z.namelist(), desc='labels ' + sub):
            if m.startswith('dataset_final/' + sub + '/labels/') and m.endswith('.txt'):
                tgt = dest / m.rsplit('/', 1)[-1]
                if not tgt.exists():
                    with z.open(m) as src, tgt.open('wb') as dst:
                        shutil.copyfileobj(src, dst)
                got += 1
        print(sub, 'labels ->', got, '| expected', (TRAIN_EXPECTED if sub == 'train' else VAL_EXPECTED))
        if got != (TRAIN_EXPECTED if sub == 'train' else VAL_EXPECTED):
            print('WARNING: label count differs from expected - the later check will tell us if any image lacks a label.')

if not CACHE_ZIPS_ON_DRIVE:
    for p in (zf, zo):
        p.unlink(missing_ok=True)
    print('Removed the downloaded zips from local disk.')
print('Labels extraction done.')



## 5. Verify dataset, write `data.yaml`, preview boxes

The class names are **derived from the label files themselves**, so the id->name mapping is always correct.


In [ ]:

def split_stats(sub):
    imgs = len(list((DATASET / sub / 'images').glob('*.jpg')))
    lbls = len(list((DATASET / sub / 'labels').glob('*.txt')))
    return imgs, lbls

tr_i, tr_l = split_stats('train')
va_i, va_l = split_stats('val')
print(f'train: {tr_i} images / {tr_l} labels')
print(f'val  : {va_i} images / {va_l} labels')
assert tr_i == tr_l, 'train labels != images'
assert va_i == va_l, 'val labels != images'

for sub in ('train', 'val'):
    stems_i = {p.stem for p in (DATASET / sub / 'images').glob('*.jpg')}
    stems_l = {p.stem for p in (DATASET / sub / 'labels').glob('*.txt')}
    miss = stems_i - stems_l
    print(sub, '-> images without label:', (sorted(miss)[:5] if miss else 0))
    if miss:
        raise RuntimeError(f'{sub}: {len(miss)} images have no label file')

# --- Derive class id -> name mapping (robust to any class ordering) ---
votes = defaultdict(Counter)
for sub in ('train', 'val'):
    for lbl in (DATASET / sub / 'labels').glob('*.txt'):
        prefix = lbl.name.split('_', 1)[0]
        first = int(lbl.read_text().split()[0])
        votes[prefix][first] += 1
mapping = {c.most_common(1)[0][0]: prefix for prefix, c in votes.items()}
nc = len(mapping)
names = [mapping[i] for i in sorted(mapping)]
print('Derived classes (id: name):', list(enumerate(names)))
assert set(mapping) == set(range(nc)), mapping

DATA_YAML = DATASET / 'data.yaml'
DATA_YAML.write_text(
    'path: ' + DATASET.as_posix() + '\n'
    'train: train/images\n'
    'val: val/images\n'
    'nc: ' + str(nc) + '\n'
    'names: ' + str(names) + '\n'
)
print(DATA_YAML.read_text())

max_id = 0
for lbl in (DATASET / 'train' / 'labels').glob('*.txt'):
    for ln in lbl.read_text().splitlines():
        if ln.strip():
            max_id = max(max_id, int(ln.split()[0]))
assert max_id < nc, max_id
print('All label class ids are within range. OK.')


In [ ]:

COLORS = plt.cm.tab10.colors
seen = {}
for img in sorted((DATASET / 'val' / 'images').glob('*.jpg')):
    pref = img.name.split('_', 1)[0]
    seen.setdefault(pref, img)
    if len(seen) == nc:
        break

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, pref in zip(axes.flat, sorted(seen)):
    img = seen[pref]
    bgr = cv2.imread(str(img))
    h, w = bgr.shape[:2]
    lbl = DATASET / 'val' / 'labels' / (img.stem + '.txt')
    for ln in lbl.read_text().splitlines():
        if not ln.strip():
            continue
        cid, x, y, bw, bh = map(float, ln.split())
        col = tuple(int(255 * c) for c in COLORS[int(cid)])
        x1 = int((x - bw / 2) * w); y1 = int((y - bh / 2) * h)
        x2 = int((x + bw / 2) * w); y2 = int((y + bh / 2) * h)
        cv2.rectangle(bgr, (x1, y1), (x2, y2), col, 2)
        cv2.putText(bgr, names[int(cid)], (x1, y1 - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, col, 2)
    ax.imshow(bgr[:, :, ::-1])
    ax.set_title(img.name)
    ax.axis('off')
plt.tight_layout()
plt.show()



## 6. Train (with automatic resume + per-epoch Drive checkpoints)

- Every finished epoch copies `last.pt` (weights + optimizer state) to `backend/training_checkpoints/`.
- If a Colab session dies, simply run the notebook again - it **auto-resumes** from that file.
- A `DONE` marker is written only after the full pipeline ends, so re-runs keep training.


In [ ]:

def copy_tree(src, dst):
    src, dst = Path(src), Path(dst)
    dst.mkdir(parents=True, exist_ok=True)
    for s in src.rglob('*'):
        if s.is_file():
            t = dst / s.relative_to(src)
            t.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(s, t)

SMALL_FILES = ('args.yaml', 'results.csv', 'results.png', 'PR_curve.png', 'P_curve.png',
               'R_curve.png', 'F1_curve.png', 'confusion_matrix.png',
               'confusion_matrix_normalized.png', 'labels.jpg', 'labels_correlogram.jpg')

def snap_run(trainer, full=False):
    sdir = Path(trainer.save_dir)
    if not sdir.exists():
        return
    if full:
        copy_tree(sdir, RUN_DIR_DRIVE)
    else:
        RUN_DIR_DRIVE.mkdir(parents=True, exist_ok=True)
        for f in SMALL_FILES:
            s = sdir / f
            if s.exists():
                shutil.copy2(s, RUN_DIR_DRIVE / f)

def checkpoint_cb(trainer):
    '''Called after every epoch: backup resume checkpoint to Drive.'''
    try:
        w = Path(trainer.save_dir) / 'weights' / 'last.pt'
        if w.exists():
            shutil.copy2(str(w), str(CKPT))
            (CKPT_DIR / 'state.json').write_text(json.dumps({
                'epoch': int(trainer.epoch),
                'total_epochs': trainer.args.epochs,
                'best_fitness': float(trainer.best_fitness or 0.0),
                'done': False,
            }, indent=2))
        snap_run(trainer, full=False)
    except Exception as e:
        print('WARN checkpoint_cb:', e)

def train_end_cb(trainer):
    '''Called when fit() finishes: sync full run + weights to Drive.'''
    try:
        snap_run(trainer, full=True)
        for wname in ('best.pt', 'last.pt'):
            w = Path(trainer.save_dir) / 'weights' / wname
            if w.exists():
                shutil.copy2(str(w), str(CKPT_DIR / wname))
        best = Path(trainer.save_dir) / 'weights' / 'best.pt'
        if best.exists():
            shutil.copy2(str(best), str(BEST_DIR / 'best.pt'))
        print('Full run + weights synced to Drive.')
    except Exception as e:
        print('WARN train_end_cb:', e)


In [ ]:

print('Device used for training:', DEVICE)

resume = False
if RESUME and CKPT.exists():
    if DONE.exists() and not FORCE_CONTINUE:
        print('DONE marker found -> training already finished.')
        print('Set FORCE_CONTINUE=True (and raise EPOCHS) to extend it, or delete the',
              'training_checkpoints folder to retrain from scratch.')
    else:
        resume = True
        print('=== AUTO-RESUME from', CKPT, '===')

if resume:
    # bring the previous epoch history back so results.csv keeps growing (not truncated)
    RUN_DIR_LOCAL.mkdir(parents=True, exist_ok=True)
    if (RUN_DIR_DRIVE / 'results.csv').exists():
        shutil.copy2(str(RUN_DIR_DRIVE / 'results.csv'), str(RUN_DIR_LOCAL / 'results.csv'))

    model = YOLO(str(CKPT))
    model.add_callback('on_fit_epoch_end', checkpoint_cb)
    model.add_callback('on_train_end', train_end_cb)
    results = model.train(resume=str(CKPT), epochs=EPOCHS, device=DEVICE, verbose=True)
else:
    model = YOLO('yolov8' + MODEL_SIZE + '.pt')   # auto-downloads COCO pretrained weights
    model.add_callback('on_fit_epoch_end', checkpoint_cb)
    model.add_callback('on_train_end', train_end_cb)
    results = model.train(
        data         = str(DATA_YAML),
        epochs       = EPOCHS,
        patience     = PATIENCE,
        imgsz        = IMGSZ,
        batch        = BATCH,
        device       = DEVICE,
        workers      = WORKERS,
        seed         = SEED,
        project      = str(RUNS_LOCAL),
        name         = 'dataset_yolov8',
        exist_ok     = True,
        cos_lr       = True,
        close_mosaic = 10,
        amp          = True,
        cache        = False,
        verbose      = True,
    )

print('Training finished at epoch', getattr(model.trainer, 'epoch', '?'))



## 7. Training curves (accuracy / loss vs epoch)


In [ ]:

CSV = RUN_DIR_DRIVE / 'results.csv'
if not CSV.exists():
    CSV = RUN_DIR_LOCAL / 'results.csv'
if not CSV.exists():
    print('No results.csv yet - run the training cell first.')
else:
    df = pd.read_csv(CSV)
    cols = list(df.columns)
    col50 = next((c for c in cols if c.startswith('metrics/mAP50') and '(B)' in c), None)
    col95 = next((c for c in cols if c.startswith('metrics/mAP50-95') and '(B)' in c), None)
    box_tr = next((c for c in cols if c.startswith('train/box_loss')), None)
    box_va = next((c for c in cols if c.startswith('val/box_loss')), None)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    if col50 and col95:
        axes[0].plot(df['epoch'], df[col50], label='val mAP50')
        axes[0].plot(df['epoch'], df[col95], label='val mAP50-95')
        best_ep = int(df.loc[df[col95].idxmax(), 'epoch'])
        print('Best epoch by mAP50-95:', best_ep)
    axes[0].set_title('Val metrics')
    axes[0].set_xlabel('epoch')
    axes[0].grid(alpha=.3)
    axes[0].legend()
    if box_tr and box_va:
        axes[1].plot(df['epoch'], df[box_tr], label='train box')
        axes[1].plot(df['epoch'], df[box_va], label='val box')
    axes[1].set_title('Box loss')
    axes[1].set_xlabel('epoch')
    axes[1].grid(alpha=.3)
    axes[1].legend()
    plt.tight_layout()
    plt.show()
    print(len(df), 'epochs logged')



## 8. Evaluate the best weights on the validation split


In [ ]:

best = RUN_DIR_LOCAL / 'weights' / 'best.pt'
if not best.exists():
    best = BEST_DIR / 'best.pt'
print('Best weights:', best, '| exists:', best.exists())

if best.exists():
    m = YOLO(str(best)).val(data=str(DATA_YAML), split='val', verbose=False, imgsz=IMGSZ)
    print('val mAP50    :', round(m.box.map50, 4))
    print('val mAP50-95 :', round(m.box.map, 4))
    for cid in range(nc):
        print('  %-12s mAP50 = %.4f' % (names[cid], m.box.maps[cid]))
    if m.box.map < 0.30:
        print('Underfit: try IMGSZ=1024 and/or a bigger model (MODEL_SIZE=m), more epochs.')
    elif m.box.map > 0.75:
        print('Strong result. Watch for a plateau (patience already handles overfitting).')
    else:
        print('Reasonable result. If you want more, try IMGSZ=1024 or MODEL_SIZE=m.')
else:
    print('Training did not complete - run the training cell first.')



## 9. Save the final model -> `backend/best/`

Saves `best.pt` + exported `best.onnx` + a `training_summary.txt` inside `backend/best/`
and marks training as `DONE` (so re-running the notebook will not overwrite it).


In [ ]:

best = RUN_DIR_LOCAL / 'weights' / 'best.pt'
if not best.exists():
    best = BEST_DIR / 'best.pt'

if not best.exists():
    print('No best.pt found - run the training cell first.')
else:
    m = YOLO(str(best))
    shutil.copy2(str(best), str(BEST_DIR / 'best.pt'))

    onnx_p = Path(m.export(format='onnx', imgsz=IMGSZ, half=False))
    shutil.copy2(str(onnx_p), str(BEST_DIR / 'best.onnx'))

    summary = []
    summary.append('SonarVision - YOLOv8' + MODEL_SIZE + ' training summary')
    summary.append('date          : ' + str(__import__('datetime').datetime.now()))
    summary.append('classes (id:name): ' + str(list(enumerate(names))))
    summary.append('num classes   : ' + str(nc))
    if 'df' in globals() and df is not None and col95 is not None and col95 in df:
        row = df.iloc[df[col95].idxmax()]
        summary.append('best epoch    : %d (mAP50-95=%.4f, mAP50=%.4f)' % (row['epoch'], row[col95], row[col50]))
        summary.append('total epochs  : %d' % len(df))
    summary.append('imgsz         : ' + str(IMGSZ))
    summary.append('batch         : ' + str(BATCH))
    summary.append('artifacts     : backend/best/best.pt, backend/best/best.onnx')
    (BEST_DIR / 'training_summary.txt').write_text('\n'.join(summary))

    (CKPT_DIR / 'state.json').write_text(json.dumps({'done': True}, indent=2))
    DONE.write_text('done ' + str(__import__('datetime').datetime.now()))

    print()
    print('Everything saved on Google Drive under:')
    print('  ' + str(BACKEND / 'runs' / 'dataset_yolov8') + '   (full run: results.csv, graphs, confusion matrix, weights)')
    print('  ' + str(CKPT_DIR) + '           (resume checkpoints)')
    print('  ' + str(BEST_DIR) + '                (best.pt + best.onnx + summary)')
    print()
    print('Saved final model:', BEST_DIR / 'best.pt')
